# SOLUTION: Tukey’s Range Test (Post-hoc after ANOVA) – VeryAnts
## Complete Interpretation, Comparison with t-tests, Simulation & Audience Reporting


## Flowchart: When and How to Use Tukey’s Range Test
```mermaid
flowchart TD
    A[Run One-way ANOVA] --> B{Is ANOVA p < 0.05?}
    B -->|Yes| C[Perform Tukey HSD<br/>pairwise_tukeyhsd(endog, groups, alpha=0.05)]
    B -->|No| Stop[Stop - No evidence of any differences]
    C --> D[Inspect output table:<br/>meandiff, lower, upper, reject, p-adj]
    D --> E{reject == True?}
    E -->|Yes| F[Significant difference between that pair<br/>Report mean diff + CI + effect size]
    E -->|No| G[No significant difference after correction]
    F --> H[Consider audience when reporting<br/>Which pairs differ? Practical importance?]
    G --> H
    H --> I[Conclusion in data analysis report style]
```
**Key advantage of Tukey:** Controls family-wise error rate for all pairwise comparisons while being more powerful than Bonferroni.


## 1. Setup and Data Loading (Solution)


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

veryants = pd.read_csv('veryants.csv')
print(veryants.head(3))
print(veryants['Store'].value_counts())


## 2. Run Tukey’s Range Test (Solution)

**Result:** Only the A vs B comparison has `reject = True`. A vs C and B vs C are not significant after Tukey correction.


In [ ]:
tukey_results = pairwise_tukeyhsd(veryants['Sale'], veryants['Store'], 0.05)
print(tukey_results)

print('\n=== Summary of significant pairs ===')
print(tukey_results.summary())


## 3. Interpretation of the Tukey Table (Solution)

**Key takeaways from the output:**
- **A vs B**: `meandiff` ≈ +7.28, 95% CI [3.23, 11.33], `reject = True` → Store B has significantly higher sales than Store A.
- **A vs C**: `meandiff` ≈ +4.01, CI crosses zero slightly, `reject = False` after correction.
- **B vs C**: Not significant.

**Comparison with previous exercise (3 separate t-tests):**
With raw t-tests we saw A vs C as significant (p=0.021). Tukey is more conservative and declares it non-significant. This is the benefit of controlling family-wise error rate.


## 4. Significance Flags (Solution)


In [ ]:
a_b_significant = True
a_c_significant = False
b_c_significant = False

print(f'A vs B significant (Tukey): {a_b_significant}')
print(f'A vs C significant (Tukey): {a_c_significant}')
print(f'B vs C significant (Tukey): {b_c_significant}')

print('\nNote: Tukey is stricter than uncorrected t-tests, protecting against false positives when doing many comparisons.')


## 5. More Practice Answers (Solution)


In [ ]:
# 1. Stricter alpha
tukey_strict = pairwise_tukeyhsd(veryants['Sale'], veryants['Store'], 0.01)
print(tukey_strict)

# 2. Comparison with Bonferroni
raw_p = [0.0000277, 0.02103, 0.05987]
reject_bonf, p_bonf, _, _ = multipletests(raw_p, method='bonferroni')
print('Bonferroni reject decisions:', reject_bonf)

# 3. Cohen's d for A vs B
a = veryants.Sale[veryants.Store == 'A']
b = veryants.Sale[veryants.Store == 'B']
pooled_sd = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
d = (b.mean() - a.mean()) / pooled_sd
print(f"Cohen's d for A vs B: {d:.3f} (medium effect)")


## 6. Simulation (Full Working Version)

When all true means are equal, Tukey correctly keeps the family-wise error rate near 5%. When real differences exist (as in our data), it has good power to detect the larger ones while protecting against over-claiming smaller differences.


In [ ]:
np.random.seed(42)

true_means = [58, 65, 62]      # try [60,60,60] for null
sigma = 15
n_per_group = 150
n_simulations = 300
alpha = 0.05

significant_count = 0

for i in range(n_simulations):
    g1 = np.random.normal(true_means[0], sigma, n_per_group)
    g2 = np.random.normal(true_means[1], sigma, n_per_group)
    g3 = np.random.normal(true_means[2], sigma, n_per_group)
    
    df_sim = pd.DataFrame({
        'value': np.concatenate([g1, g2, g3]),
        'group': ['A']*n_per_group + ['B']*n_per_group + ['C']*n_per_group
    })
    
    tukey_sim = pairwise_tukeyhsd(df_sim['value'], df_sim['group'], alpha)
    
    if any(tukey_sim.reject):
        significant_count += 1

detection_rate = significant_count / n_simulations
label = 'Power (detecting real differences)' if not np.allclose(true_means, true_means[0]) else 'Family-wise error rate (false positive)'
print(f'{label}: {detection_rate:.3f}')
print('Tukey maintains good error control while still detecting meaningful differences.')


## 7. Example Conclusion & Audience Reporting (Solution)

### Overall Conclusion
Tukey’s range test following a significant one-way ANOVA showed that only Store B had significantly higher average sales than Store A (mean difference ≈ $7.28, 95% CI [3.23, 11.33], p-adj < 0.001). The differences between A vs C and B vs C were not statistically significant after correction for multiple comparisons. This is more conservative than running three separate t-tests (which had flagged A vs C as significant).

**Practical takeaway:** Store B is the clear outperformer relative to Store A. Further investigation into location-specific factors at B is recommended.

### Audience-tailored versions

**For Executives:**
> "According to Tukey’s test, Store B has significantly higher average sales than Store A (about $7 more per sale). We did not find clear differences between the other pairs after accounting for multiple comparisons. We should study what Store B is doing well."

**For Technical team:**
> "Tukey HSD after significant ANOVA (F=8.96, p=0.00015): only A vs B significant (meandiff=7.28, reject=True). A vs C and B vs C not significant after FWER correction. More conservative than uncorrected t-tests. Cohen’s d for A vs B ≈ 0.49 (medium)."

**For Non-technical stakeholders:**
> "We used a careful statistical test (Tukey) that checks all store pairs at once while avoiding false alarms. It confirmed that Store B really does have higher average sales than Store A. The other comparisons were not strong enough to be sure after our correction."
